# NB08 - Weak Supervision & Labeling Functions
## Background
Manual labeling at scale is expensive. Weak supervision (Ratner et al., 2017 — the Snorkel system) generates noisy labels programmatically from multiple weak signals: heuristic rules, domain knowledge, pre-trained classifiers, and knowledge bases. A label model then combines these signals, learning their accuracies and correlations, to produce final probabilistic labels.

The key insight: multiple imperfect labeling functions (LFs) can together outperform a single perfect annotator if their errors are sufficiently diverse. Snorkel is used in production at Google, Intel, and Stanford Hospital. GPT-4 as a labeling function is a modern variant.

## What You'll Learn
- Labeling functions: heuristic, keyword-based, model-based
- Label model: majority vote and weighted majority vote
- LF analysis: coverage, accuracy, conflict rates
- End-model training on probabilistic weak labels

## Why This Matters
Weak supervision enables building labeled datasets in hours instead of months. For new domains where ground truth is expensive (medical imaging, legal NLP, industrial QA), it is often the only practical path to supervised learning at scale.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Callable, Tuple, Optional

# ── Simulate a text classification task ───────────────────────────────────
np.random.seed(42)
n_samples = 1000
n_classes = 3  # Sentiment: negative, neutral, positive
NEGATIVE, NEUTRAL, POSITIVE = 0, 1, 2
ABSTAIN = -1

# Ground truth (hidden during weak supervision)
y_true = np.random.choice([NEGATIVE, NEUTRAL, POSITIVE],
                           size=n_samples, p=[0.35, 0.3, 0.35])

# Feature: "sentiment score" that correlates with true label + noise
base_scores = np.where(y_true == POSITIVE, 1.0, np.where(y_true == NEGATIVE, -1.0, 0.0))
sentiment_score = base_scores + np.random.randn(n_samples) * 0.8

# Simulated other features
has_positive_keyword = (base_scores > 0) & (np.random.rand(n_samples) > 0.3)
has_negative_keyword = (base_scores < 0) & (np.random.rand(n_samples) > 0.3)
text_length          = np.random.randint(10, 500, n_samples)
domain_indicator     = np.random.choice([0, 1, 2], n_samples)  # Domain

# Combine into feature matrix for later
X_features = np.column_stack([
    sentiment_score, has_positive_keyword.astype(float),
    has_negative_keyword.astype(float), text_length / 500.0,
    domain_indicator / 2.0
])

print(f'Dataset: {n_samples} unlabeled samples with features')
print(f'True label distribution: {dict(zip(["neg","neu","pos"], np.bincount(y_true)))}')

In [ ]:
# ── Define labeling functions ──────────────────────────────────────────────
def lf_sentiment_threshold_high(X, threshold=0.8):
    """LF1: High sentiment score -> positive."""
    votes = np.full(len(X), ABSTAIN)
    votes[X[:, 0] > threshold] = POSITIVE
    return votes

def lf_sentiment_threshold_low(X, threshold=-0.8):
    """LF2: Low sentiment score -> negative."""
    votes = np.full(len(X), ABSTAIN)
    votes[X[:, 0] < threshold] = NEGATIVE
    return votes

def lf_positive_keyword(X):
    """LF3: Has positive keyword -> positive."""
    votes = np.full(len(X), ABSTAIN)
    votes[X[:, 1] > 0.5] = POSITIVE
    return votes

def lf_negative_keyword(X):
    """LF4: Has negative keyword -> negative."""
    votes = np.full(len(X), ABSTAIN)
    votes[X[:, 2] > 0.5] = NEGATIVE
    return votes

def lf_neutral_short(X):
    """LF5: Short text + near-zero sentiment -> neutral."""
    votes = np.full(len(X), ABSTAIN)
    short = X[:, 3] < 0.1
    mid_sent = np.abs(X[:, 0]) < 0.5
    votes[short & mid_sent] = NEUTRAL
    return votes

def lf_domain_specific(X):
    """LF6: Domain 2 tends to be positive reviews."""
    votes = np.full(len(X), ABSTAIN)
    votes[X[:, 4] > 0.9] = POSITIVE
    return votes

LFs = [
    ('LF1_high_sentiment',  lf_sentiment_threshold_high),
    ('LF2_low_sentiment',   lf_sentiment_threshold_low),
    ('LF3_pos_keyword',     lf_positive_keyword),
    ('LF4_neg_keyword',     lf_negative_keyword),
    ('LF5_neutral_short',   lf_neutral_short),
    ('LF6_domain',          lf_domain_specific),
]

# Apply all LFs to get label matrix L
L = np.column_stack([fn(X_features) for _, fn in LFs])  # (n_samples, n_LFs)

print('Labeling function matrix L shape:', L.shape)
print()
print(f'{'LF Name':25s} | {'Coverage':>10} | {'Accuracy':>10} | {'Abstain':>10}')
print('-' * 60)
for i, (name, _) in enumerate(LFs):
    coverage = (L[:, i] != ABSTAIN).mean()
    labeled_mask = L[:, i] != ABSTAIN
    acc = (L[labeled_mask, i] == y_true[labeled_mask]).mean() if labeled_mask.sum() > 0 else 0
    abstain = 1 - coverage
    print(f'{name:25s} | {coverage:>10.3f} | {acc:>10.3f} | {abstain:>10.3f}')

In [ ]:
# ── Label model: weighted majority vote ───────────────────────────────────
def majority_vote_label_model(L: np.ndarray, n_classes: int) -> np.ndarray:
    """Simple majority vote ignoring ABSTAIN."""
    probs = np.zeros((len(L), n_classes))
    for i in range(len(L)):
        votes = L[i][L[i] != ABSTAIN]
        if len(votes) == 0:
            probs[i] = 1.0 / n_classes
        else:
            for v in votes:
                probs[i, int(v)] += 1
            probs[i] /= len(votes)
    return probs

def weighted_majority_vote(L: np.ndarray, y_true: np.ndarray,
                            n_classes: int) -> np.ndarray:
    """Weight each LF by its accuracy on available data."""
    n_lfs = L.shape[1]
    weights = np.zeros(n_lfs)
    for j in range(n_lfs):
        mask = L[:, j] != ABSTAIN
        if mask.sum() > 0:
            acc = (L[mask, j] == y_true[mask]).mean()
            weights[j] = max(acc - 0.33, 0.01)  # Accuracy above chance

    probs = np.zeros((len(L), n_classes))
    for i in range(len(L)):
        for j in range(n_lfs):
            if L[i, j] != ABSTAIN:
                probs[i, int(L[i, j])] += weights[j]
        if probs[i].sum() > 0:
            probs[i] /= probs[i].sum()
        else:
            probs[i] = 1.0 / n_classes
    return probs

# Generate probabilistic labels
probs_mv  = majority_vote_label_model(L, n_classes)
probs_wmv = weighted_majority_vote(L, y_true, n_classes)

y_mv  = np.argmax(probs_mv, axis=1)
y_wmv = np.argmax(probs_wmv, axis=1)

acc_mv  = (y_mv == y_true).mean()
acc_wmv = (y_wmv == y_true).mean()

print('Label Model Performance:')
print(f'  Majority Vote:          {acc_mv:.3f}')
print(f'  Weighted Majority Vote: {acc_wmv:.3f}')
print(f'  Random baseline:        {1/n_classes:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(L.T, aspect='auto', cmap='RdYlGn', vmin=-1, vmax=2)
axes[0].set_title('Label Matrix L (rows=LFs, cols=samples)')
axes[0].set_xlabel('Sample index'); axes[0].set_ylabel('Labeling function')
axes[0].set_yticks(range(len(LFs)))
axes[0].set_yticklabels([n for n,_ in LFs], fontsize=8)

for c in range(n_classes):
    label = ['Negative', 'Neutral', 'Positive'][c]
    axes[1].plot(sorted(probs_wmv[:, c]), label=f'{label} prob', linewidth=1.5)
axes[1].set_title('Sorted Probabilistic Labels (Weighted MV)')
axes[1].set_xlabel('Sample rank'); axes[1].set_ylabel('Class probability')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE}/s29_08_weak_supervision.png', dpi=100, bbox_inches='tight')
plt.show()